## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE 
This script creates the phenotype file for SAIGE-GENE+. It integrates several inputs, including EUR-like exome sample quality control results, ancestry-specific PCs, and age and sex files. Only individuals who pass exome quality control are retained in the SAIGE-GENE+ phenotype file.
Case–control status is assigned by providing a list of person IDs for individuals designated as cases; all remaining individuals are assigned as controls. The script also restricts the dataset to individuals who have electronic health record data. All quantitative covariates are standardized to have a mean of 0 and a standard deviation of 1.
Because more samples passed exome quality control than ACAF quality control in this analysis, this phenotype file will also be used for the ACAF analysis, with samples failing ACAF quality control excluded at a later stage.

In [29]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

In [30]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [31]:
# Load quality control information and demographic information 
eur_pca = pd.read_csv(f"{my_bucket}/data/pcs/EUR_flashpca.pcs", 
                      sep ='\t')
eur_exome = pd.read_csv(f"{my_bucket}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt", 
                        sep ='\t')
biosample = pd.read_csv(f"{my_bucket}/data/covariates/biosample_collection_age.tsv", 
                        sep ='\t')
age = pd.read_csv(f"{my_bucket}/data/covariates/observation_age_period.tsv", 
                  sep ='\t')
sex = pd.read_csv(f"{my_bucket}/data/wgs_qc/ancestry_sd3_wgs_qc/EUR-like_cohort_info_qc.txt", 
                  sep ='\t')

In [32]:
# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

In [33]:
import pandas
import os

# This query represents dataset "aml_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39009800_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (140352)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (4304199) 
                    AND is_standard = 1 )) criteria ) )"""

aml_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/774 [00:00<?, ?rows/s]

In [34]:
import pandas
import os

# This query represents dataset "electronic_health_records" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_72216074_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

ehr_person_df = pandas.read_gbq(
    dataset_72216074_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/393601 [00:00<?, ?rows/s]

In [35]:
# Create list of quality control information and demographic information 
dfs = [eur_exome.rename(columns={"IID":"person_id"}),
       age.rename(columns={"IID":"person_id"}).filter(regex="person_id|age"),
       biosample.rename(columns={"IID":"person_id", 
                                 "age":"biosample"}).filter(regex="person_id|biosample"),
       sex.rename(columns={"IID":"person_id"}).filter(regex="person_id|sex"), 
       eur_pca.rename(columns={"IID":"person_id"}).filter(regex="person_id|PC"),
       ehr_person_df
       ]

In [36]:
# Combine list of quality control information and demographic information 
phenotype_file = reduce(
    lambda left, right: pd.merge(left, right, on="person_id", how="inner"),
    dfs
)

In [37]:
# Assign AML phenotype to individuals 
phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

In [38]:
# Format phenotype file  
phenotype_file = (phenotype_file[['aml_phenotype','sex', 'age', 'biosample',
            'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 
            'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 
            'person_id']]
)

In [39]:
median_biosample = phenotype_file['biosample'].median()

In [40]:
# Age restriction 
restricted_file = phenotype_file[phenotype_file['biosample'] < median_biosample]
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col in restricted_file.columns if col.startswith("PC") or col.startswith("age")]
# Standardize only the selected columns
scaler = StandardScaler()
restricted_file[cols_to_standardize] = scaler.fit_transform(restricted_file[cols_to_standardize])
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(restricted_file["aml_phenotype"].value_counts())
print("============================================")
# Save phenoytpe file
bucket = f"{my_bucket}/data/saige_phenotype_files/aml_biosample_lower_50th_percentile_files/EUR_phenotype_file.txt"
restricted_file.to_csv(bucket, sep ='\t', index=None)

# Age restriction 
restricted_file = phenotype_file[phenotype_file['biosample'] >= median_biosample]
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col in restricted_file.columns if col.startswith("PC") or col.startswith("age")]
# Standardize only the selected columns
scaler = StandardScaler()
restricted_file[cols_to_standardize] = scaler.fit_transform(restricted_file[cols_to_standardize])
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(restricted_file["aml_phenotype"].value_counts())
print("============================================")
# Save phenoytpe file
bucket = f"{my_bucket}/data/saige_phenotype_files/aml_biosample_upper_50th_percentile_files/EUR_phenotype_file.txt"
restricted_file.to_csv(bucket, sep ='\t', index=None)

/tmp/ipykernel_102/2060892659.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  restricted_file[cols_to_standardize] = scaler.fit_transform(restricted_file[cols_to_standardize])


Check the presence of aml phenotype:
aml_phenotype
0    84894
1       90
Name: count, dtype: int64


/tmp/ipykernel_102/2060892659.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  restricted_file[cols_to_standardize] = scaler.fit_transform(restricted_file[cols_to_standardize])


Check the presence of aml phenotype:
aml_phenotype
0    84812
1      175
Name: count, dtype: int64
